# LightGBM Hyperparameter Tuning

## Business Objective

The objective of this notebook is to improve the baseline LightGBM model by tuning its hyperparameters using Time Series Cross Validation.

Unlike random cross-validation, time-series validation preserves chronological order and prevents data leakage, making it suitable for forecasting problems.

In [131]:
# ==========================================================
# Hyperparameter Tuning
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from lightgbm import LGBMRegressor

from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

In [132]:
# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("../data/burger_data_engineered.csv")

df["Date"] = pd.to_datetime(df["Date"])

df.head()

,Date,Region,Temperature,Humidity,Wind,Visibility,Pressure,Sales,Year,Month,Day,DayOfWeek,Quarter,WeekOfYear,IsWeekend,Sales_Lag_1,Sales_Lag_7,Sales_Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7
0,2014-01-03,Reg2,18.174600,0.435980,24.228756,13.744171,1016.652700,992.37,2014,1,3,4,1,1,0,204.32,699.35,1539.45,1581.357143,1639.600333,1291.820530
1,2014-01-04,Reg10,21.861622,0.402545,11.281557,4.461844,1023.733652,23.83,2014,1,4,5,1,1,1,992.37,2774.98,2721.81,1623.217143,1621.364333,1262.900440
2,2014-01-04,Reg8,3.776241,0.886265,5.661404,9.982186,1020.505489,672.21,2014,1,4,5,1,1,1,23.83,950.34,2240.22,1230.195714,1531.431667,1272.774252
3,2014-01-04,Reg3,0.196468,0.829441,5.494188,8.846484,1024.717837,664.99,2014,1,4,5,1,1,1,672.21,405.83,468.78,1190.462857,1479.164667,1287.225940
4,2014-01-04,Reg9,23.463183,0.724208,10.477588,11.057665,1010.657532,2765.08,2014,1,4,5,1,1,1,664.99,3087.19,137.76,1227.485714,1485.705000,1264.422258


In [ ]:
encoder = joblib.load("../models/region_encoder.pkl")
scaler = joblib.load("../models/quantile_scaler.pkl")

try:
    df["Region"] = encoder.transform(df["Region"])
except ValueError:
    region_digits = df["Region"].astype(str).str.extract(r"(\d+)", expand=False)
    if region_digits.isna().any():
        df["Region"], _ = pd.factorize(df["Region"])
    else:
        df["Region"] = encoder.transform(region_digits.astype(int))


_IncompleteInputError: incomplete input (3408453707.py, line 14)

In [ ]:
X = df.drop(
    columns=[
        "Sales",
        "Date"
    ]
)

y = df["Sales"]
train_size = int(len(df) * 0.70)

validation_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]
X_validation = X.iloc[
    train_size:
    train_size + validation_size
]
X_test = X.iloc[
    train_size + validation_size:
]

y_train = y.iloc[:train_size]
y_validation = y.iloc[
    train_size:
    train_size + validation_size
]
y_test = y.iloc[
    train_size + validation_size:
]